# Aula 7 — BERT e HuggingFace

Disciplina **CCM-109 Deep Learning** — Prof. Ronaldo Prati (UFABC)

**Tópicos:**
1. MLM Demo — preenchendo lacunas com BERT
2. Embeddings de texto — semântica no espaço vetorial
3. Embeddings não-textuais — Sentence-BERT, CLIP e dados tabulares
4. Visualização de atenção BERT
5. Fine-tuning para classificação de sentimento (SST-2)

In [ ]:
!pip install transformers datasets sentence-transformers torch torchvision --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print(f'Transformers: ', end=''); import transformers; print(transformers.__version__)

---
## Parte 1 — MLM Demo: BERT prevê tokens mascarados

O pré-treinamento do BERT usa **Masked Language Modeling (MLM)**:
- 15% dos tokens são substituídos por `[MASK]`
- O modelo prevê o token original usando contexto **bidirecional** (esquerda e direita)

Vamos explorar o BERT pré-treinado fazendo previsões de lacunas.

In [ ]:
from transformers import pipeline

fill_mask = pipeline('fill-mask', model='bert-base-uncased',
                     device=0 if torch.cuda.is_available() else -1)

In [ ]:
def show_predictions(text, top_k=5):
    results = fill_mask(text, top_k=top_k)
    print(f'Texto: "{text}"\n')
    print(f'{"Token":20s}  {"Score":>8s}  Frase')
    print('-' * 60)
    for r in results:
        print(f"{r['token_str']:20s}  {r['score']:8.4f}  {r['sequence']}")
    print()

# Contexto desambigua o sentido de "bank"
show_predictions("She went to the bank to [MASK] money.")
show_predictions("She sat on the river [MASK] and watched the sunset.")

In [ ]:
# Visualização: distribuição de probabilidade
text = "Python is a popular programming [MASK]."
results = fill_mask(text, top_k=10)
tokens = [r['token_str'] for r in results]
scores = [r['score'] for r in results]

fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(tokens)))
bars = ax.barh(tokens[::-1], scores[::-1], color=colors[::-1])
ax.set_xlabel('Probabilidade')
ax.set_title(f'BERT fill-mask: "{text}"', pad=10)
for bar, score in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=9)
ax.set_xlim(0, max(scores) * 1.25)
plt.tight_layout(); plt.show()

---
## Parte 2 — Embeddings de texto com BERT

BERT gera **embeddings contextualizados**: o mesmo token terá vetores diferentes dependendo do contexto.
Usaremos mean pooling sobre os tokens para obter uma representação global da sentença.

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name).to(device)
bert.eval()
print('BERT carregado.')

In [ ]:
def get_bert_embedding(texts, pooling='mean'):
    """Extrai embeddings de sentenças usando BERT."""
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = bert(**inputs)
    last_hidden = outputs.last_hidden_state  # (B, T, 768)
    if pooling == 'cls':
        return last_hidden[:, 0, :].cpu().numpy()
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    return (last_hidden * mask).sum(1).div(mask.sum(1)).cpu().numpy()

In [ ]:
sentences = [
    'Python is a programming language used for machine learning.',
    'Deep learning requires large amounts of training data.',
    'Neural networks learn hierarchical representations.',
    'The soccer team won the championship last night.',
    'Basketball players need excellent coordination and speed.',
    'The marathon runner finished the race in record time.',
    'The pasta was cooked with olive oil and fresh tomatoes.',
    'Baking bread requires flour, yeast, water, and salt.',
    'The chef prepared a delicious chocolate cake for dessert.',
]
labels_cat = ['Tech']*3 + ['Sport']*3 + ['Food']*3
colors_map = {'Tech': '#3b82f6', 'Sport': '#ef4444', 'Food': '#10b981'}

embs = get_bert_embedding(sentences, pooling='mean')
print(f'Shape dos embeddings: {embs.shape}')

In [ ]:
pca = PCA(n_components=2)
embs_2d = pca.fit_transform(embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, (x, y) in enumerate(embs_2d):
    color = colors_map[labels_cat[i]]
    ax.scatter(x, y, color=color, s=120, zorder=3)
    ax.annotate(sentences[i][:35]+'…', (x, y), textcoords='offset points',
                xytext=(8, 4), fontsize=7)
for cat, col in colors_map.items():
    ax.scatter([], [], color=col, label=cat, s=80)
ax.legend(); ax.set_title('PCA 2D — BERT mean pooling')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.grid(alpha=0.3)

ax2 = axes[1]
embs_norm = normalize(embs)
sim_matrix = embs_norm @ embs_norm.T
short = [s[:30]+'…' for s in sentences]
im = ax2.imshow(sim_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0)
ax2.set_xticks(range(9)); ax2.set_yticks(range(9))
ax2.set_xticklabels(short, rotation=45, ha='right', fontsize=7)
ax2.set_yticklabels(short, fontsize=7)
for i in range(9):
    for j in range(9):
        ax2.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax2); ax2.set_title('Similaridade coseno')
plt.tight_layout(); plt.show()

---
## Parte 3 — Embeddings não-textuais

### 3.1 Sentence-BERT — embeddings otimizados para similaridade

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2')

query = "How to train a machine learning model?"
candidates = [
    "Steps to fit a neural network on data.",
    "The weather was sunny and warm today.",
    "Training procedures for deep learning systems.",
    "My dog loves playing fetch in the park.",
    "Gradient descent optimizes model parameters.",
]

q_bert  = normalize(get_bert_embedding([query]))
c_bert  = normalize(get_bert_embedding(candidates))
q_sbert = normalize(sbert.encode([query]))
c_sbert = normalize(sbert.encode(candidates))

sim_bert  = (q_bert  @ c_bert.T)[0]
sim_sbert = (q_sbert @ c_sbert.T)[0]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(candidates)); w = 0.35
ax.bar(x - w/2, sim_bert,  w, label='BERT mean pool', color='#3b82f6', alpha=0.8)
ax.bar(x + w/2, sim_sbert, w, label='Sentence-BERT',  color='#10b981', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([c[:35]+'…' for c in candidates], rotation=18, ha='right', fontsize=8)
ax.set_ylabel('Similaridade coseno'); ax.set_ylim(0, 1)
ax.set_title(f'Query: "{query}"')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### 3.2 CLIP — texto e imagem no mesmo espaço vetorial

CLIP (Radford et al., 2021) aprende embeddings conjuntos de texto e imagem, permitindo classificação zero-shot.

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

image_urls = {
    'dog':   'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg',
    'cat':   'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Image_created_with_a_mobile_phone.png/320px-Image_created_with_a_mobile_phone.png',
    'pizza': 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg/320px-Eq_it-na_pizza-margherita_sep2005_sml.jpg',
}
images = {}
for name, url in image_urls.items():
    try:
        resp = requests.get(url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        images[name] = Image.open(BytesIO(resp.content)).convert('RGB')
        print(f'Carregada: {name}')
    except Exception as e:
        print(f'Erro: {name} — {e}')

In [ ]:
text_candidates = [
    'a photo of a dog', 'a photo of a cat', 'a photo of a pizza',
    'a photo of a car', 'a photo of a forest',
]

if images:
    fig, axes = plt.subplots(1, len(images), figsize=(14, 5))
    for ax, (img_name, img) in zip(axes, images.items()):
        inputs = clip_proc(text=text_candidates, images=img,
                           return_tensors='pt', padding=True).to(device)
        with torch.no_grad():
            logits = clip_model(**inputs).logits_per_image[0]
        probs = logits.softmax(dim=0).cpu().numpy()
        ax.imshow(img); ax.axis('off')
        title = f'{img_name.upper()}\n'
        for t, p in sorted(zip(text_candidates, probs), key=lambda x: -x[1])[:3]:
            title += f'{t}: {p:.1%}\n'
        ax.set_title(title, fontsize=8)
    plt.suptitle('CLIP: classificação zero-shot texto→imagem', fontsize=11)
    plt.tight_layout(); plt.show()

### 3.3 Embeddings tabulares com PyTorch

Variáveis categóricas são mapeadas para vetores densos com `nn.Embedding`.
Os vetores são inicializados aleatoriamente e **aprendidos durante o treino** junto com o restante da rede.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42); torch.manual_seed(42)
N = 500

plano_idx     = torch.randint(0, 3, (N,))   # basic=0, standard=1, premium=2
regiao_idx    = torch.randint(0, 5, (N,))   # 5 regiões
pagamento_idx = torch.randint(0, 3, (N,))   # credit=0, debit=1, boleto=2

meses = torch.FloatTensor(np.random.exponential(12, N).clip(1, 60)) / 60
valor = torch.FloatTensor(np.random.normal(150, 50, N).clip(50, 300)) / 300
numericas = torch.stack([meses, valor], dim=1)

# Label sintética: plano básico + boleto = mais churn
churn_prob = 0.3*(plano_idx==0).float() + 0.2*(pagamento_idx==2).float() + 0.1*torch.rand(N)
labels = (churn_prob > 0.38).float()
print(f'Dataset: {N} clientes | Taxa churn: {labels.mean():.1%}')

In [ ]:
class ChurnModel(nn.Module):
    def __init__(self, emb_dim=8):
        super().__init__()
        self.emb_plano     = nn.Embedding(3, emb_dim)
        self.emb_regiao    = nn.Embedding(5, emb_dim)
        self.emb_pagamento = nn.Embedding(3, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*3 + 2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),            nn.ReLU(),
            nn.Linear(32, 1),             nn.Sigmoid()
        )

    def forward(self, plano, regiao, pagamento, num):
        x = torch.cat([
            self.emb_plano(plano),
            self.emb_regiao(regiao),
            self.emb_pagamento(pagamento),
            num
        ], dim=1)
        return self.mlp(x).squeeze(1)

churn_model = ChurnModel(emb_dim=8).to(device)
optimizer_c = torch.optim.Adam(churn_model.parameters(), lr=1e-3)
criterion_c = nn.BCELoss()

ds = TensorDataset(plano_idx, regiao_idx, pagamento_idx, numericas, labels)
loader_c = DataLoader(ds, batch_size=64, shuffle=True)

for epoch in range(1, 31):
    churn_model.train()
    total_loss = 0
    for batch in loader_c:
        p, r, pg, num_b, y = [b.to(device) for b in batch]
        pred = churn_model(p, r, pg, num_b)
        loss = criterion_c(pred, y)
        optimizer_c.zero_grad(); loss.backward(); optimizer_c.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f'Epoch {epoch:2d} | loss: {total_loss/len(loader_c):.4f}')

In [ ]:
# Visualizar embeddings aprendidos por PCA
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
churn_model.eval()

configs = [
    (churn_model.emb_plano,     ['basic','standard','premium'], 'Plano'),
    (churn_model.emb_regiao,    ['N','S','L','O','C'],          'Região'),
    (churn_model.emb_pagamento, ['credit','debit','boleto'],    'Pagamento'),
]
for ax, (layer, cats, name) in zip(axes, configs):
    vecs = layer.weight.detach().cpu().numpy()
    pts  = PCA(n_components=2).fit_transform(vecs)
    colors = plt.cm.Set1(np.linspace(0, 0.8, len(cats)))
    for i, (cat, col) in enumerate(zip(cats, colors)):
        ax.scatter(*pts[i], color=col, s=250, zorder=3)
        ax.annotate(cat, pts[i], textcoords='offset points',
                    xytext=(8, 4), fontsize=10, fontweight='bold')
    ax.set_title(f'{name} — embeddings aprendidos')
    ax.grid(alpha=0.3); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Embeddings tabulares (pesos após treino — PCA 2D)', fontsize=10, y=1.01)
plt.tight_layout(); plt.show()

---
## Parte 4 — Visualização de atenção BERT

Extraímos os pesos de atenção com `output_attentions=True` e criamos heatmaps para analisar quais tokens "prestam atenção" em quais.

In [ ]:
from transformers import BertModel

bert_attn = BertModel.from_pretrained('bert-base-uncased',
                                       output_attentions=True).to(device)
bert_attn.eval()

def get_attention(text, layer=11):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    with torch.no_grad():
        outputs = bert_attn(**inputs)
    attn = outputs.attentions[layer][0].cpu().numpy()  # (12, T, T)
    return tokens, attn

sentence = "The bank can guarantee deposits will eventually cover future tuition costs."
tokens, attn_last = get_attention(sentence, layer=11)
print(f'Tokens: {tokens}')
print(f'Shape atenção (última camada): {attn_last.shape}')

In [ ]:
# 6 cabeças da última camada
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for h, ax in enumerate(axes.flatten()):
    im = ax.imshow(attn_last[h], cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'Cabeça {h+1}')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle('Pesos de atenção — Camada 12 (6 cabeças)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Comparação: camada 1 vs camada 12 (média das cabeças)
_, attn_first = get_attention(sentence, layer=0)
_, attn_last  = get_attention(sentence, layer=11)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (attn, title) in zip(axes, [
    (attn_first.mean(0), 'Camada 1 — média das 12 cabeças'),
    (attn_last.mean(0),  'Camada 12 — média das 12 cabeças'),
]):
    im = ax.imshow(attn, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.suptitle(f'"{sentence}"', fontsize=9, y=1.01)
plt.tight_layout(); plt.show()

print('Camadas iniciais: atenção local (tokens adjacentes)')
print('Camadas profundas: relações semânticas de longa distância')

---
## Parte 5 — Fine-tuning para classificação de sentimento (SST-2)

BERT + cabeça linear, fine-tuning completo com lr=2e-5 e warmup de 10%.

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

dataset = load_dataset('nyu-mll/glue', 'sst2')
print(dataset)
print('\nAmostra:')
for ex in dataset['train'].select(range(3)):
    print(f"  [{'pos' if ex['label'] else 'neg'}] {ex['sentence'][:70]}")

In [ ]:
class SSTDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=64):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(item['sentence'], max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(item['label'], dtype=torch.long)
        }

train_ds = SSTDataset(dataset['train'].shuffle(seed=42).select(range(2000)), tokenizer)
val_ds   = SSTDataset(dataset['validation'], tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
EPOCHS = 3; LR = 2e-5

clf_model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2).to(device)

encoder_params = [p for n, p in clf_model.named_parameters() if 'classifier' not in n]
head_params    = [p for n, p in clf_model.named_parameters() if 'classifier' in n]
optimizer = AdamW([
    {'params': encoder_params, 'lr': LR},
    {'params': head_params,    'lr': LR * 10},
], weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
print(f'Parâmetros: {sum(p.numel() for p in clf_model.parameters()):,}')

In [ ]:
from torch.amp import GradScaler, autocast
scaler = GradScaler(enabled=torch.cuda.is_available())

def train_epoch(model, loader):
    model.train()
    loss_sum = correct = total = 0
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        y     = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast(device_type=device.type, enabled=torch.cuda.is_available()):
            out = model(input_ids=ids, attention_mask=mask, labels=y)
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        preds = out.logits.argmax(-1)
        loss_sum += out.loss.item() * y.size(0)
        correct  += (preds == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y    = batch['labels'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=y)
        preds = out.logits.argmax(-1)
        loss_sum += out.loss.item() * y.size(0)
        correct  += (preds == y).sum().item()
        total    += y.size(0)
    return loss_sum / total, correct / total

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
for epoch in range(1, EPOCHS + 1):
    tl, ta = train_epoch(clf_model, train_loader)
    vl, va = evaluate(clf_model, val_loader)
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'Epoch {epoch}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS + 1)
ax1.plot(ep, history['train_loss'], 'o-', label='Train', color='#3b82f6')
ax1.plot(ep, history['val_loss'],   's-', label='Val',   color='#ef4444')
ax1.set(xlabel='Época', ylabel='Loss', title='Loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history['train_acc'], 'o-', label='Train', color='#3b82f6')
ax2.plot(ep, history['val_acc'],   's-', label='Val',   color='#ef4444')
ax2.set(xlabel='Época', ylabel='Acurácia', title='Acurácia', ylim=(0,1))
ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Fine-tuning BERT — SST-2', fontsize=11); plt.tight_layout(); plt.show()

In [ ]:
def predict_sentiment(texts):
    clf_model.eval()
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=64).to(device)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs = logits.softmax(-1).cpu().numpy()
    for text, prob in zip(texts, probs):
        pred = 'positivo' if prob.argmax() == 1 else 'negativo'
        print(f'[{pred} {prob.max():.2%}] {text[:70]}')

predict_sentiment([
    "This movie was absolutely fantastic and I loved every minute of it!",
    "The film was a complete waste of time, terribly boring.",
    "It was an okay movie, nothing special but not terrible either.",
    "One of the best performances I have ever seen on screen.",
])

---
## Resumo

| Tópico | API usada |
|---|---|
| **MLM demo** | `pipeline('fill-mask')` |
| **Embeddings texto** | `AutoModel` + mean pooling |
| **Sentence-BERT** | `SentenceTransformer('all-MiniLM-L6-v2')` |
| **CLIP** | `CLIPModel` + `CLIPProcessor` |
| **Embeddings tabulares** | `nn.Embedding` treinável em MLP |
| **Atenção BERT** | `BertModel(output_attentions=True)` |
| **Fine-tuning SST-2** | `AutoModelForSequenceClassification` + `AdamW` + warmup |

**Leituras:** Devlin et al. (2018) BERT · Radford et al. (2021) CLIP · Reimers & Gurevych (2019) Sentence-BERT